# Rich Neale's Blocking Diagnostic (ESNB version)

### How to run
The run settings are in two text cells in Section 1, written in the same formats as the MDTF files:
the runtime config (`input_*.yml`) and the POD settings (`settings.jsonc`). The notebook detects how it is being run:

- **Interactive (Jupyter)**: edit those two cells and run all cells. The export cell writes them out as
  `input_<case>.yml` and `settings.jsonc` and prints the commands to repeat the run in batch.
  The last cell writes the POD web page `<WORK_DIR>/blocking_neale_nb/blocking_neale_nb.html`.
- **Headless (no Jupyter)**, from this directory:
  `MDTF_RUNTIME_CONFIG=<input.yml> [MDTF_POD_SETTINGS=<settings.jsonc>] jupyter nbconvert --to html --execute blocking_main_esnb.ipynb --output-dir <WORK_DIR>/blocking_neale_nb --output blocking_main_esnb_ipynb`
  (without `MDTF_POD_SETTINGS`, `settings.jsonc` in this directory is used). Also writes the POD web page.
- **MDTF framework**: `mdtf -f <input.yml>` with `blocking_neale_nb` in `pod_list`. The framework reads
  `settings.jsonc` from this directory and builds the POD web page itself.


The POD is built upon the following procedures:

### Section 1: Prepare POD Settings
Set the parameters of the POD
Set the runtime configurations
### Section 2: Load Data from Catalog Files
Two default options available (users can override)
Here: ESNB option for catalog reading
TBA : Intake-ESM option

### Section 3: Diagnostics Computation

### Section 4: Results Visualization
### Section 5: More about this POD
Check the .rst file
Placeholder

More details on the development process:
[MDTF Planning Document](https://docs.google.com/document/d/1P8HqL8O5304qwR3ik9RmgFDwSWwlkPgOjnp39PIkLfY/edit?usp=sharing)

[ESNB GitHub Repository](https://github.com/jkrasting/esnb/tree/ncar-updates)

In [2]:
# Import General Package

import importlib

import sys
import yaml
import json

import os


General Settings


In [3]:


# Give your diagnostic a name and a short description
diag_name = "Blocking (Neale) Notebook-based Diagnostic"
diag_desc = "This example demonstrates how to use the notebook template"


In [ ]:
#
# Settings for how this Notebook is being used
#
# The run mode (interactive / headless / framework) is detected automatically
# in the "Run settings" cell below.

usage = "dev"
# usage:
#         dev  (development): either CESM time series files with no pre-processing done
#                             OR pre-processed files already made, used directly
#         prod (production) : MDTF full functionality.
#                             Any supported data type is provided through catalogs and translated as necessary
#                             by the MDTF pre-processor.
#

machine = "casper-notdask"
# machine:
#         Uses machine-specfic things such as dask.
#         This should be determined by the framework
#

test = 0
# test:
#         test = 1: find and load data, skip analysis/plotting
#         test = 0: run the full analysis and plottings
#

# Verbosity
verbose = True


In [5]:
# Development usage: constantly refreshes module code

if (usage == "dev"):  
    %load_ext autoreload
    %autoreload 2


In [6]:
#
# Machine Specific Settings
# These are particular to NCAR's Casper HPC system. For other systems, add your own settings here.
#

if (machine ==  "casper"):
    print("Using Casper-specific szettings")
    # Use dask jobqueue
    from dask_jobqueue import PBSCluster

    # Import a client
    from dask.distributed import Client

    #P03010039. #P93300642

    cluster = PBSCluster(cores=4, memory="8GB", account='P03010039', interface='ext', job_extra_directives=[], walltime='2:00:00', queue='casper', resource_spec='select=1:ncpus=4:mem=8GB')
    cluster
    cluster.scale(jobs=1)
    client = Client(cluster)
    client

#Ideally this then waits to confirm the job has started, since it goes through the queuing system. Current usage: use qstat -u $USER in the terminal to find it.

## Framework Code and Diagnostic Setup

In [ ]:
#
# Framework: Setup paths to import MDTF code as modules
#
POD_NAME = "blocking_neale_nb"
NOTEBOOK = "blocking_main_esnb.ipynb"

# POD code directory: $POD_HOME when run by the MDTF framework (which runs a copy of this
# notebook in the output directory), otherwise the directory this notebook is in.
pod_code_dir = os.path.abspath(os.environ.get("POD_HOME", "."))
module_path = os.path.abspath(os.path.join(pod_code_dir, "../../"))  # MDTF-diagnostics/
print(f"Using MDTF framework version {module_path}")
for p in (module_path, pod_code_dir):
    if p not in sys.path:
        sys.path.append(p)
from src import util              #, varlist_util, translation, xr_parser, units
from src.util import json_utils
from src import standalone_html   # run settings, export, and stand-alone POD web page

import blocking_utils as block_utils
import blocking_figs as block_figs


In [8]:

# The ESNB package has  notebook-specific classes and functions
import esnb
from esnb import NotebookDiagnostic, RequestedVariable, CaseGroup2

os.environ["ESNB_LOG_LEVEL"] = "DEBUG"

In [ ]:
#
# POD settings, in settings.jsonc format (JSON with // comments).
# Used when running interactively. Headless and framework runs read settings.jsonc
# from the POD directory instead; the export cell below writes this text to a file.
#
pod_settings_jsonc = r"""
// MDTF POD settings for blocking_neale_nb (Rich Neale's blocking diagnostic, notebook version)
//
// JSON with "//" comments (JSONC). The same text is held in the notebook's
// POD settings cell; the notebook's export cell writes it back out as this file.
{
  "settings" : {
    // Notebook run by the framework (jupyter nbconvert --execute)
    "driver" : "blocking_main_esnb.ipynb",
    "long_name" : "Rich Neale's blocking diagnostic",
    "convention": "cesm",
    "description" : "Rich Neale's blocking diagnostic (notebook version)",

    // POD options. The framework passes these to the notebook as environment
    // variables, so every value is a string; lists are comma-separated.
    "pod_env_vars" : {
      // Season for the blocking frequency: DJF, MAM, JJA, SON or ANN
      "MDTF_BLOCKING_SEASON"      : "DJF",
      // Hemisphere(s): nhem, shem
      "MDTF_BLOCKING_HEMISPHERE"  : "nhem",
      // Obs and model ensembles to compare with the MDTF case(s)
      "MDTF_BLOCKING_ENSEMBLES"   : "ERA5,MERRA,CESM2",
      // Members to use from each ensemble above (same order)
      "MDTF_BLOCKING_ENS_MEMBERS" : "1,1,2",
      // Years for the obs and model ensembles. Blank: use the MDTF case years.
      // (The NCL version of this POD compared against 1979-1999.)
      "MDTF_BLOCKING_REF_FIRSTYR" : "",
      "MDTF_BLOCKING_REF_LASTYR"  : ""
    },

    "runtime_requirements": {
      "python3": []
    }
  },

  "dimensions": {
    "lat": {
      "standard_name": "latitude",
      "units": "degrees_north",
      "axis": "Y"
    },
    "lon": {
      "standard_name": "longitude",
      "units": "degrees_east",
      "axis": "X"
    },
    "lev": {
      "standard_name": "air_pressure",
      "units": "hPa",
      "positive": "down",
      "axis": "Z"
    },
    "time": {"standard_name": "time"}
  },

  "varlist" : {
    "zg": {
      "path_variable" : "MODEL_DATA_PATH",
      "standard_name" : "geopotential_height",
      "units": "m",
      "realm" : "atmos",
      "frequency" : "day",
      "dimensions": ["time", "lat", "lon"],
      "scalar_coordinates": {"lev" : 500}
    }
  }
}
"""


In [ ]:
#
# Run settings, in MDTF runtime config (input_*.yml) format.
# Used when running interactively. A headless run reads the file named by
# $MDTF_RUNTIME_CONFIG instead; the export cell below writes this text to a file.
#
runtime_yml = r"""
# List of POD(s) to run (used by the MDTF framework)
pod_list:
   - "blocking_neale_nb"

# Case list entries (must be unique IDs for each simulation)
case_list:
  "timeslice_mdtfv3" :
    model: "CESM"
    convention: "CESM"
    startdate: "19950101000000"
    enddate:   "19961231000000"

### Data location settings ###
# Required: full or relative path to ESM-intake catalog header file
DATA_CATALOG: "/glade/u/home/bundy/diag/mdtf/catalogs/esm_catalog_CESM_timeslice_mdtfv3.20241107.json"
# Optional: Parent directory containing observational data used by individual PODs.
OBS_DATA_ROOT: "/glade/work/bundy/mdtf/inputdata/obs_data"
# Required: Working directory location. The POD output goes in WORK_DIR/blocking_neale_nb
WORK_DIR: "/glade/work/bundy/mdtf/outdir/blocking_nb"
# Optional: Location to write final output if you don't want it in the WORK_DIR
OUTPUT_DIR: ""

### Environment Settings (used by the MDTF framework) ###
conda_root: "/glade/work/bundy/miniconda2/"
conda_env_root: "/glade/work/bundy/miniconda2/envs.MDTFv4.2.20260505/"
micromamba_exe: ""

### Data type settings ###
large_file: False

### Output Settings ###
# Set to true to have PODs save postscript figures in addition to bitmaps.
save_ps: False
save_pp_data: True
translate_data: True
make_variab_tar: True    # also write <WORK_DIR>/blocking_neale_nb.tar.gz (page + figures + banner)
overwrite: True
make_multicase_figure_html: False
### Developer settings ###
run_pp: True
user_pp_scripts:
  - ""
"""


In [ ]:
#
# Resolve the run settings for this run mode and show them
#   interactive : the two text cells above
#   headless    : $MDTF_RUNTIME_CONFIG file, and $MDTF_POD_SETTINGS or ./settings.jsonc
#   framework   : the MDTF framework's environment variables and case_info.yml
#
run = standalone_html.load_run_settings(POD_NAME, pod_code_dir,
                                        runtime_yml_text=runtime_yml,
                                        pod_settings_text=pod_settings_jsonc)
settings_dict = run.settings
case_info = run.runtime_cfg
pod_env = run.pod_env_vars
print(f"Run mode: {run.mode}\n")
print(yaml.dump(case_info, sort_keys=False))
print(json.dumps(pod_env, indent=2))

# POD options (pod_env_vars are strings; lists are comma-separated)
block_season = pod_env["MDTF_BLOCKING_SEASON"]
block_diag_hem = [s.strip() for s in pod_env["MDTF_BLOCKING_HEMISPHERE"].split(",")]
ens_names = [s.strip() for s in pod_env["MDTF_BLOCKING_ENSEMBLES"].split(",")]
ens_mem_num = [int(s) for s in pod_env["MDTF_BLOCKING_ENS_MEMBERS"].split(",")]
assert len(ens_names) == len(ens_mem_num), "MDTF_BLOCKING_ENSEMBLES and MDTF_BLOCKING_ENS_MEMBERS differ in length"

# Case period (all cases are assumed to cover the same period)
def _iso_date(d):
    d = str(d)
    return f"{d[0:4]}-{d[4:6]}-{d[6:8]}" if d.isdigit() else d[:10]
first_case = next(iter(case_info["case_list"].values()))
start_year = standalone_html.year_of(first_case["startdate"])
end_year = standalone_html.year_of(first_case["enddate"])
date_range = (_iso_date(first_case["startdate"]), _iso_date(first_case["enddate"]))

# Years for the obs and model ensembles (blank: same as the case)
ref_start_year = pod_env.get("MDTF_BLOCKING_REF_FIRSTYR", "").strip() or start_year
ref_end_year = pod_env.get("MDTF_BLOCKING_REF_LASTYR", "").strip() or end_year

# Output follows the MDTF layout: <WORK_DIR>/blocking_neale_nb/model/ (figures) and model/netCDF/
pod_work_dir = run.pod_work_dir
standalone_html.make_pod_work_dirs(pod_work_dir)
fig_dir = os.path.join(pod_work_dir, "model")
fout_dir = os.path.join(pod_work_dir, "model", "netCDF")
print(f"\nSeason {block_season}, hemisphere {block_diag_hem}, case years {start_year}-{end_year}, "
      f"obs/ensemble years {ref_start_year}-{ref_end_year}")
print(f"Ensembles {ens_names}, members {ens_mem_num}")
print(f"Figures -> {fig_dir}\nnetCDF  -> {fout_dir}")


In [ ]:
#
# Export the run settings so this run can be repeated in batch (interactive runs only).
# Writes input_<case>.yml and settings.jsonc to the POD output directory and prints the commands.
# install_pod_settings = True also copies settings.jsonc into the POD code directory,
# which is where the MDTF framework reads it (overwrites that file; it is in git).
#
export_settings = True
install_pod_settings = False
if export_settings and run.mode == "interactive":
    exported = standalone_html.export_run_settings(
        run, notebook=os.path.join(pod_code_dir, NOTEBOOK),
        install_pod_settings=install_pod_settings)


In [10]:
# We should ask developers to generate a settings file before submitting. But while developing, it is easier to use the settings_dict directly.

#  Initialize the diagnostic with its name, description, vars, and options
# This is the setttings file from mdtf/diagnostics/$DIAG_NAME/settings.jsonc (Must be full path)
#test diag = NotebookDiagnostic("/glade/u/home/bundy/diag/mdtf/esnb/esnb/src/esnb/data/settings.jsonc")
diag = NotebookDiagnostic(settings_dict)
#print(diag)
from pprint import pprint
pprint(vars(diag))

{'_observers': {},
 '_settings_keys': ['description',
                    'convention',
                    'driver',
                    'long_name',
                    'pod_env_vars',
                    'runtime_requirements'],
 'convention': 'cesm',
 'description': "Rich Neale's blocking diagnostic",
 'diag_vars': {},
 'dimensions': {'lat': {'axis': 'Y',
                        'standard_name': 'latitude',
                        'units': 'degrees_north'},
                'lev': {'axis': 'Z',
                        'positive': 'down',
                        'standard_name': 'air_pressure',
                        'units': 'hPa'},
                'lon': {'axis': 'X',
                        'standard_name': 'longitude',
                        'units': 'degrees_east'},
                'time': {'standard_name': 'time'}},
 'driver': 'blocking_main.ipynb',
 'groups': [],
 'long_name': "Rich Neale's blocking diagnostic",
 'name': "Rich Neale's blocking diagnostic",
 'pod_env_vars': {

In [11]:
#Variable name, and possibly other information. This could be deduced from the mdtf/data/fieldlist_$MODEL.json files
mapping = {
    "zg": "Z500",
}


In [ ]:
#
# Give the case info to the ESNB CaseGroup2 class
#
mdtf_groups = [
    CaseGroup2(
        case_info,
        date_range=date_range,
        mapping=mapping,
    )
]


In [14]:
mdtf_groups[0]

source,"{'case_list': {'timeslice_mdtfv3': {'convention': 'CESM', 'model': 'CESM', 'startdate': '19950101000000', 'enddate': '19961231000000'}}, 'DATA_CATALOG': '/glade/u/home/bundy/diag/mdtf/catalogs/esm_catalog_CESM_timeslice_mdtfv3.20241107.json', 'OBS_DATA_ROOT': '/glade/work/bundy/mdtf/inputdata/obs_data', 'WORK_DIR': '/glade/work/bundy/mdtf/outdir/20250429/blocking_nb', 'OUTPUT_DIR': '/glade/work/bundy/mdtf/outdir/20250429/blocking_nb', 'conda_env_root': '/glade/work/bundy/miniconda2/envs.MDTF.20241107/', 'conda_root': '/glade/work/bundy/miniconda2/', 'large_file': False, 'make_multicase_figure_html': False, 'make_variab_tar': False, 'micromamba_exe': '', 'overwrite': True, 'run_pp': True, 'save_pp_data': True, 'save_ps': False, 'translate_data': True, 'user_pp_scripts': ['']}"
date_range,"('1995-01-01', '1996-12-31')"
concat_dim,None
is_resolved,False
is_loaded,False
plot_color,None
cases,[CaseExperiment2(timeslice_mdtfv3)]


In [15]:
#Inspect the one case more carefully
mdtf_groups[0].cases[0]

Source Type,dictionary
catalog,esm_catalog_CESM_timeslice_mdtfv3.20241107 catalog with 3 dataset(s) from 38 asset(s)
DATA_CATALOG,/glade/u/home/bundy/diag/mdtf/catalogs/esm_catalog_CESM_timeslice_mdtfv3.20241107.json
OBS_DATA_ROOT,/glade/work/bundy/mdtf/inputdata/obs_data
OUTPUT_DIR,/glade/work/bundy/mdtf/outdir/20250429/blocking_nb
WORK_DIR,/glade/work/bundy/mdtf/outdir/20250429/blocking_nb
case_list,"{'timeslice_mdtfv3': {'convention': 'CESM', 'model': 'CESM', 'startdate': '19950101000000', 'enddate': '19961231000000'}}"
conda_env_root,/glade/work/bundy/miniconda2/envs.MDTF.20241107/
conda_root,/glade/work/bundy/miniconda2/
large_file,False
make_multicase_figure_html,False


In [16]:
#
# ESNB Function Read the catalog, rename variables, find files to open
diag.resolve(mdtf_groups)

In [17]:
# Inspect which files are actually being read (as found through the catalog)
diag.files

['/glade/campaign/cgd/amp/bundy/mdtf/cesm_mdtfv3_timeslice_public/atm/day/cesm_mdtfv3_timeslice.Z500.day.nc']

In [18]:
#Open the model data files using ESNB function
diag.open()

In [19]:
# Show what datasets are available
diag.datasets

[<xarray.Dataset> Size: 161MB
 Dimensions:  (lat: 192, lon: 288, time: 730)
 Coordinates:
   * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
   * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
   * time     (time) object 6kB 1995-01-01 00:00:00 ... 1996-12-31 00:00:00
 Data variables:
     Z500     (time, lat, lon) float32 161MB dask.array<chunksize=(84, 91, 137), meta=np.ndarray>
 Attributes:
     Conventions:       CF-1.0
     source:            CAM
     case:              bhist.e213cosp217.f09_g17.mdtf_v3.1995
     logname:           bundy
     host:              cheyenne5
     initial_file:      bhist.e213cosp217.f09_g17.mdtf_v3.1993.cam.i.1995-01-0...
     topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...
     model_doi_url:     https://doi.org/10.5065/D67H1H0V
     time_period_freq:  day_1]

<i>(The files above are necessary to run the diagnostic.)</i>

In [20]:
diag.variables[0]

RequestedVariable Z500

## Example: Exploring Datasets by Looping Over Variables

In [21]:
# First loop over variables, and then over groups

for variable in diag.variables:
    for group in variable.datasets.keys():
        ds = variable.datasets[group]
        print("\n")
        print(f"Variable={variable}, Group={group}")
        print(ds)



Variable=Z500, Group=timeslice_mdtfv3
<xarray.Dataset> Size: 161MB
Dimensions:  (lat: 192, lon: 288, time: 730)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * time     (time) object 6kB 1995-01-01 00:00:00 ... 1996-12-31 00:00:00
Data variables:
    Z500     (time, lat, lon) float32 161MB dask.array<chunksize=(84, 91, 137), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              bhist.e213cosp217.f09_g17.mdtf_v3.1995
    logname:           bundy
    host:              cheyenne5
    initial_file:      bhist.e213cosp217.f09_g17.mdtf_v3.1993.cam.i.1995-01-0...
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1


In [22]:
#See some actual numbers, just checking that the data is there
#print(ds["Z500"].values[:10, :10, :10])  # Print first 10 values for Z500 variable

## Example: Exploring Datasets by Looping over Groups

In [23]:
# First loop over groups, and then over variables

for group in diag.groups:
    for variable in group.datasets.keys():
        ds = group.datasets[variable]
        print("\n")
        print(f"Variable={variable}, Group={group}")
        print(ds)



Variable=Z500, Group=timeslice_mdtfv3
<xarray.Dataset> Size: 161MB
Dimensions:  (lat: 192, lon: 288, time: 730)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * time     (time) object 6kB 1995-01-01 00:00:00 ... 1996-12-31 00:00:00
Data variables:
    Z500     (time, lat, lon) float32 161MB dask.array<chunksize=(84, 91, 137), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              bhist.e213cosp217.f09_g17.mdtf_v3.1995
    logname:           bundy
    host:              cheyenne5
    initial_file:      bhist.e213cosp217.f09_g17.mdtf_v3.1993.cam.i.1995-01-0...
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1


# Blocking Code Main Routines


### Load OBS data 

In [24]:

#### ...Ensemble comparison cases...

ensemble_configs = {
    "CESM1": {
        "ens_type": "model",
        "dir_ens0": "/glade/campaign/cesm/collections/cesmLE/CESM-CAM5-BGC-LE/atm/proc/tseries/",
        "dir_day_add": "daily",
        "file_template": lambda run: f"/glade/campaign/cesm/collections/cesmLE/CESM-CAM5-BGC-LE/atm/proc/tseries/daily/VAR_TBD/{run}.cam.h1.VAR_TBD.19200101-20051231.nc",
        "special_case": lambda run, file: file.replace('1920', '1850', 1) if run == 'b.e11.B20TRC5CNBDRD.f09_g16.001' else file
    },
    "CESM2": {
        "ens_type": "model",
        "dir_ens0": "/glade/campaign/cgd/cesm/CESM2-LE/atm/proc/tseries/",
        "dir_day_add": "day_1",
        "file_template": lambda run: f"/glade/campaign/cgd/cesm/CESM2-LE/atm/proc/tseries/day_1/VAR_TBD/{run}.cam.h1.VAR_TBD.DATE_RANGE.nc"
    },
    "E3SMv1": {
        "ens_type": "model",
        "dir_ens0": "/glade/campaign/cgd/amp/rneale/e3sm/",
        "dir_day_add": "day_1",
        "file_template": lambda run: f"/glade/campaign/cgd/amp/rneale/e3sm/{run}/tseries/{run}_dmeans_ts_VAR_TBD.nc"
    },
    "E3SMv2": {
        "ens_type": "model",
        "cmodel": "eam",
        "dir_ens0": "/glade/campaign/cgd/ccr/E3SMv2/FV_regridded/",
        "dir_day_add": "day_1",
        "file_template": lambda run: f"/glade/campaign/cgd/ccr/E3SMv2/FV_regridded/{run}/atm/proc/tseries/day_1/{run}.eam.h1.VAR_TBD.18500101-20141231.nc"
    },
    "EAMv2": {
        "ens_type": "model",
        "cmodel": "eam",
        "dir_ens0": "/glade/campaign/cgd/ccr/E3SMv2/FV_regridded/",
        "dir_day_add": "day_1",
        "file_template": lambda run: f"/glade/campaign/cgd/ccr/E3SMv2/FV_regridded/{run}/atm/proc/tseries/day_1/{run}.eam.h1.VAR_TBD.19760101-20141231.nc"
    },
    "CAM6": {
        "ens_type": "model",
        "cmodel": "cam",
        "dir_ens0": "/glade/campaign/cesm/development/cvcwg/cvwg/f.e21.FHIST_FSSP370_BGC.f09_f09.ersstv5.goga/",
        "dir_day_add": "day_1",
        "file_template": lambda run: f"/glade/campaign/cesm/development/cvcwg/cvwg/f.e21.FHIST_FSSP370_BGC.f09_f09.ersstv5.goga/{run}/atm/proc/tseries/day_1/{run}.cam.h1.VAR_TBD.18800101-20150101.nc"
    },
    "b.e23": {
        "ens_type": "model",
        "dir_ens0": "/glade/derecho/scratch/hannay/archive/",
        "file_template": lambda run: f"/glade/derecho/scratch/hannay/archive/{run}/atm/hist/{run}.cam.h1..nc"
    },
    # Add obs sources dynamically
}

# Example obs_sources list
obs_sources = ['ERA5', 'MERRA', 'CESM2']

for obs in obs_sources:
    ensemble_configs[obs] = {
        "ens_type": "obs",
        "dir_ens0": f"/glade/work/rneale/data/{obs}/",
        "file_template": lambda run=None: f"/glade/work/rneale/data/{obs}/VAR_TBD.day.mean.nc"
    }

# Usage example:
ens_name = "CESM1"
run_names = ["b.e11.B20TRC5CNBDRD.f09_g16.001", "b.e11.B20TRC5CNBDRD.f09_g16.002"]
config = ensemble_configs[ens_name]
file_templates = [config["file_template"](run) for run in run_names]
if "special_case" in config:
    file_templates = [config["special_case"](run, file) for run, file in zip(run_names, file_templates)]

print(file_templates)
# ...existing code...



['/glade/campaign/cesm/collections/cesmLE/CESM-CAM5-BGC-LE/atm/proc/tseries/daily/VAR_TBD/b.e11.B20TRC5CNBDRD.f09_g16.001.cam.h1.VAR_TBD.18500101-20051231.nc', '/glade/campaign/cesm/collections/cesmLE/CESM-CAM5-BGC-LE/atm/proc/tseries/daily/VAR_TBD/b.e11.B20TRC5CNBDRD.f09_g16.002.cam.h1.VAR_TBD.19200101-20051231.nc']


In [25]:
print(ensemble_configs)

{'CESM1': {'ens_type': 'model', 'dir_ens0': '/glade/campaign/cesm/collections/cesmLE/CESM-CAM5-BGC-LE/atm/proc/tseries/', 'dir_day_add': 'daily', 'file_template': <function <lambda> at 0x7f27e440fd80>, 'special_case': <function <lambda> at 0x7f27e1e4df80>}, 'CESM2': {'ens_type': 'obs', 'dir_ens0': '/glade/work/rneale/data/CESM2/', 'file_template': <function <lambda> at 0x7f27e1e4e520>}, 'E3SMv1': {'ens_type': 'model', 'dir_ens0': '/glade/campaign/cgd/amp/rneale/e3sm/', 'dir_day_add': 'day_1', 'file_template': <function <lambda> at 0x7f27e1e4de40>}, 'E3SMv2': {'ens_type': 'model', 'cmodel': 'eam', 'dir_ens0': '/glade/campaign/cgd/ccr/E3SMv2/FV_regridded/', 'dir_day_add': 'day_1', 'file_template': <function <lambda> at 0x7f27e1e4e160>}, 'EAMv2': {'ens_type': 'model', 'cmodel': 'eam', 'dir_ens0': '/glade/campaign/cgd/ccr/E3SMv2/FV_regridded/', 'dir_day_add': 'day_1', 'file_template': <function <lambda> at 0x7f27e1e4e200>}, 'CAM6': {'ens_type': 'model', 'cmodel': 'cam', 'dir_ens0': '/glade

In [ ]:
### This is from Rich's original, but I think it should work here. 
### Check if it repeats what is done above, and/or is used at all!
###       Eventually pull the ens_setup to the cell level though so user can modify

### Read in obs/ens data
### block_meta type: <class 'pandas.core.frame.DataFrame'>
### block_meta columns:
### ['Ensemble Type', 'Ensemble Size', 'Start Year', 'End Year', 'Run Name', 'Run File'],
### block_meta index: Index(['ERA5', 'MERRA', 'CESM2'], dtype='object')

# ens_names, ens_mem_num, block_season, block_diag_hem and the obs/ensemble years
# (ref_start_year, ref_end_year) are set in the run settings cell (Section 1)
importlib.reload(block_utils) # Update for any edits to utility routines.
ens_ystart = [ref_start_year]*len(ens_names)
ens_yend   = [ref_end_year]*len(ens_names)
cam_var = 'Z500' #Where are the obs variable names set?
block_diag_set = ['1d']

# Grab basic ens+run information and populate a dataframe
block_meta = block_utils.ens_setup(ens_names,ens_mem_num,ens_ystart,ens_yend)


In [27]:
#See what is in the block_meta structure

print("block_meta type:", type(block_meta))
print("block_meta columns:", block_meta.columns if hasattr(block_meta, 'columns') else None)
print("block_meta index:", block_meta.index if hasattr(block_meta, 'index') else None)
print("block_meta head:")
print(block_meta.head())
print("block_meta info:")
if hasattr(block_meta, 'info'):
    block_meta.info()

block_meta type: <class 'pandas.core.frame.DataFrame'>
block_meta columns: Index(['Ensemble Type', 'Ensemble Size', 'Start Year', 'End Year', 'Run Name',
       'Run File'],
      dtype='object')
block_meta index: Index(['ERA5', 'MERRA', 'CESM2'], dtype='object')
block_meta head:
      Ensemble Type  Ensemble Size Start Year End Year  \
ERA5            obs              1       1995     1996   
MERRA           obs              1       1995     1996   
CESM2         model              2       1995     1996   

                                                Run Name  \
ERA5                                              [ERA5]   
MERRA                                            [MERRA]   
CESM2  [b.e21.BHISTcmip6.f09_g17.LE2-1001.001, b.e21....   

                                                Run File  
ERA5   [/glade/work/rneale/data/ERA5/VAR_TBD.day.mean...  
MERRA  [/glade/work/rneale/data/MERRA/VAR_TBD.day.mea...  
CESM2  [/glade/campaign/cgd/cesm/CESM2-LE/atm/proc/ts...  
block_met

In [ ]:

print(f"\n Reading files from table below")
print(f"\n Writing output to {fout_dir}")

# Grab basic ens+run information and populate a dictionary)
block_meta = block_utils.ens_setup(ens_names,ens_mem_num,ens_ystart,ens_yend)

print(f"block_meta keys: {block_meta.keys()}")
print(f"block_meta: {block_meta}")


### Extract Model (new case(s)) data from structure

In [29]:
# Extract the MDTF group/case information and add to the pandas structure
import pandas as pd

for group in diag.groups: 
    print(f"Group: {group.name}")
    n_cases = len(group.datasets)
    print(f"Found {n_cases=} in group")
    print(f"  Group items {group.cases=}")

    for case in group.cases:
        print(f"Case attributes for {case.name}:")
        #for attr, value in vars(case).items():
        #fails on Nans in realm column
        #    print(f"  {attr}: {value}")
            
        print(f"    name: {case.name}")
        print(f"    Files: {case.files()}")
        print(f"    source: {case.source}")        
        #print(f"    all mdtf_settings: {case.mdtf_settings}")
        
        #Example on how to get something out of the structure
        mdtf_settings = case.mdtf_settings
        
        #All settings 
        case_list = mdtf_settings.get('case_list')
        print(f"    {case_list=}") 
        
        for case_name, case_details in case_list.items():
            print(f"Model: {case_details.get('model')}")
            print(f"Convention: {case_details.get('convention')}")
            print(f"Start Date: {case_details.get('startdate')}")
            print(f"End Date: {case_details.get('enddate')}")


    # Add to the dictionary
    new_row = {
        'Ensemble Type': 'mdtf',
        'Ensemble Size': n_cases,
        'Start Year': case_details['startdate'], #these are all the same for the group
        'End Year': case_details['enddate'],     #these are all the same for the group
        'Run Name': case_name, #need to do multiple 
        'Run File': diag.files #how does this work with the multiple cases, even groups??
        # could pass xarrays instead of file names: diag.datasets[group.name]   
        }
    new_row_df = pd.DataFrame([new_row], index=[case_name])
    #print(new_row_df)

    block_meta = pd.concat([block_meta, new_row_df])
    

print(f' {block_meta=}')




Group: timeslice_mdtfv3
Found n_cases=1 in group
  Group items group.cases=[CaseExperiment2(timeslice_mdtfv3)]
Case attributes for timeslice_mdtfv3:
    name: timeslice_mdtfv3
    Files: ['/glade/campaign/cgd/amp/bundy/mdtf/cesm_mdtfv3_timeslice_public/atm/day/cesm_mdtfv3_timeslice.Z500.day.nc']
    source: {'case_list': {'timeslice_mdtfv3': {'convention': 'CESM', 'model': 'CESM', 'startdate': '19950101000000', 'enddate': '19961231000000'}}, 'DATA_CATALOG': '/glade/u/home/bundy/diag/mdtf/catalogs/esm_catalog_CESM_timeslice_mdtfv3.20241107.json', 'OBS_DATA_ROOT': '/glade/work/bundy/mdtf/inputdata/obs_data', 'WORK_DIR': '/glade/work/bundy/mdtf/outdir/20250429/blocking_nb', 'OUTPUT_DIR': '/glade/work/bundy/mdtf/outdir/20250429/blocking_nb', 'conda_env_root': '/glade/work/bundy/miniconda2/envs.MDTF.20241107/', 'conda_root': '/glade/work/bundy/miniconda2/', 'large_file': False, 'make_multicase_figure_html': False, 'make_variab_tar': False, 'micromamba_exe': '', 'overwrite': True, 'run_pp': 

In [ ]:
# block_season and block_diag_hem are set in the run settings cell (Section 1)
block_diag_set = ['1d']


# Grab daily datasets needed for each 'ensemble'

In [31]:
cam_var = 'Z500'  #The name of the variable in the digested data, not necessarily the variable in the catalog
ensemble_ds = block_utils.dataset_get(block_meta,cam_var,block_season,block_diag_hem)

-> dataset_get -> Requested season     :  DJF
-> dataset_get ->  == Starting ensemble:  0 ERA5 Type: <class 'list'>
-> dataset_get ->  Opening run_files: ['/glade/work/rneale/data/ERA5/VAR_TBD.day.mean.nc']
-> dataset_get ->  Opening ensemble files ERA5  -  1  ensemble(s)
-> dataset_get -> Requested year range :  1995 - 1996
-> dataset_get ->   Opening file:  0 /glade/work/rneale/data/ERA5/Z500.day.mean.nc ensemble:  ERA5
-> dataset_get ->  == Starting ensemble:  1 MERRA Type: <class 'list'>
-> dataset_get ->  Opening run_files: ['/glade/work/rneale/data/MERRA/VAR_TBD.day.mean.nc']
-> dataset_get ->  Opening ensemble files MERRA  -  1  ensemble(s)
-> dataset_get -> Requested year range :  1995 - 1996
-> dataset_get ->   Opening file:  0 /glade/work/rneale/data/MERRA/Z500.day.mean.nc ensemble:  MERRA
-> dataset_get ->  Finished opening <xarray.Dataset> Size: 9GB
Dimensions:  (time: 12053, lon: 540, lat: 361)
Coordinates:
  * time     (time) float64 96kB 2.444e+06 2.444e+06 ... 2.456e+06

/glade/work/bundy/miniconda2/envs.MDTFv4.2.20260505/_MDTF_python3_base/lib/python3.12/site-packages/dask/utils.py:77: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  return func(*args, **kwargs)
/glade/work/bundy/miniconda2/envs.MDTFv4.2.20260505/_MDTF_python3_base/lib/python3.12/site-packages/dask/utils.py:77: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  return func(*args, **kwargs)
/glade/work/bundy/miniconda2/envs.MDTFv4.2.20260505/_MDTF_python3_base/lib/python3.12/site-packages/dask/utils.py:77: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  return func(*args, **kwargs)
/gla

-> dataset_get ->   Opening file:  1 /glade/campaign/cgd/cesm/CESM2-LE/atm/proc/tseries/day_1/Z500/b.e21.BHISTcmip6.f09_g17.LE2-1021.002.cam.h1.Z500.DATE_RANGE.nc ensemble:  CESM2


/glade/work/bundy/miniconda2/envs.MDTFv4.2.20260505/_MDTF_python3_base/lib/python3.12/site-packages/dask/utils.py:77: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  return func(*args, **kwargs)
/glade/work/bundy/miniconda2/envs.MDTFv4.2.20260505/_MDTF_python3_base/lib/python3.12/site-packages/dask/utils.py:77: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  return func(*args, **kwargs)
/glade/work/bundy/miniconda2/envs.MDTFv4.2.20260505/_MDTF_python3_base/lib/python3.12/site-packages/dask/utils.py:77: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  return func(*args, **kwargs)
/gla

-> dataset_get ->  == Starting ensemble:  3 timeslice_mdtfv3 Type: <class 'list'>
-> dataset_get ->  Opening run_files: ['/glade/campaign/cgd/amp/bundy/mdtf/cesm_mdtfv3_timeslice_public/atm/day/cesm_mdtfv3_timeslice.Z500.day.nc']
-> dataset_get ->  Opening ensemble files timeslice_mdtfv3  -  16  ensemble(s)
-> dataset_get -> Requested year range :  1995 - 1996
-> dataset_get ->   Opening file:  0 /glade/campaign/cgd/amp/bundy/mdtf/cesm_mdtfv3_timeslice_public/atm/day/cesm_mdtfv3_timeslice.Z500.day.nc ensemble:  timeslice_mdtfv3
-> dataset_get ->  Finished opening <xarray.Dataset> Size: 2GB
Dimensions:    (lat: 192, lon: 288, time: 7300, nbnd: 2)
Coordinates:
  * lat        (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon        (lon) float64 2kB 0.0 1.25 2.5 3.75 ... 355.0 356.2 357.5 358.8
  * time       (time) object 58kB 1995-01-01 00:00:00 ... 2014-12-31 00:00:00
Dimensions without coordinates: nbnd
Data variables:
    time_bnds  (time, nbnd) object 117kB d

/glade/work/bundy/miniconda2/envs.MDTFv4.2.20260505/_MDTF_python3_base/lib/python3.12/site-packages/dask/utils.py:77: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  return func(*args, **kwargs)


#####  Calculate/Read/Write blocking frequency: 1D

Summarize the datasets that are loaded

In [32]:
#print(ensemble_ds)
ens_names = list(block_meta.index)
print(f"Ensemble names: {ens_names}")

#i = 1
#ens_name = ens_names[i]
#print(f"Changing name of ensemble  {ens_name=}")
#ensemble_ds[ens_name].coords["name"] = ens_name


for i in range(len(ens_names)):
    ens_name = ens_names[i]
    print(f"++++ Ensemble {i}: {ens_name}")
    print(f"      {ensemble_ds[ens_name].dims}")


#one_ds = 
#print(f"one_ds: {one_ds}")


Ensemble names: ['ERA5', 'MERRA', 'CESM2', 'timeslice_mdtfv3']
++++ Ensemble 0: ERA5
      FrozenMappingWarningOnValuesAccess({'time': 731, 'bnds': 2, 'lon': 360, 'lat': 180, 'name': 1})
++++ Ensemble 1: MERRA
      FrozenMappingWarningOnValuesAccess({'time': 731, 'lon': 540, 'lat': 361, 'name': 1})
++++ Ensemble 2: CESM2
      FrozenMappingWarningOnValuesAccess({'name': 2, 'time': 1461, 'lat': 551, 'lon': 792, 'lev': 32, 'ilev': 33, 'nbnd': 2})
++++ Ensemble 3: timeslice_mdtfv3
      FrozenMappingWarningOnValuesAccess({'lat': 192, 'lon': 288, 'time': 730, 'nbnd': 2, 'name': 1})


In [33]:
# Either calculate and write a file with, or read in the blocking frequency
# The file_opts = 'x' means to check if the file exists, and if not, calculate and write it out. If it does exist, read it in.
#
#TODO: we don't need the original case files (specified above) unless we aren't reading existing analyses.

importlib.reload(block_utils) # Update for any edits to utility routines.  #Should be automatic

ensemble_block_1d = block_utils.block_z500_freq(block_meta,ensemble_ds,fout_dir,block_season,block_diag = '1D', file_opts = 'w')

-> block_z500_freq ->  Starting freq blocking calculation
-> block_z500_freq ->   Calculating blocking statistics for  ERA5
--> block_file_read_write ->  Writing file  /glade/work/bundy/mdtf/outdir/20250429/blocking_nb/block_1D_ERA5_nens.1_1995-1996_DJF.nc
--> block_file_read_write ->  Done ...
-> block_z500_freq ->  Min/max blocking frequency for ensemble  ERA5  =  1.6415868673050615 , 25.718194254445965
-> block_z500_freq ->   Calculating blocking statistics for  MERRA
--> block_file_read_write ->  Writing file  /glade/work/bundy/mdtf/outdir/20250429/blocking_nb/block_1D_MERRA_nens.1_1995-1996_DJF.nc
--> block_file_read_write ->  Done ...
-> block_z500_freq ->  Min/max blocking frequency for ensemble  MERRA  =  1.5047879616963065 , 24.623803009575923
-> block_z500_freq ->   Calculating blocking statistics for  CESM2
--> block_file_read_write ->  Writing file  /glade/work/bundy/mdtf/outdir/20250429/blocking_nb/block_1D_CESM2_nens.2_1995-1996_DJF.nc
--> block_file_read_write ->  Done .


##### Calculate/Read/Write blocking frequency: 2D


In [34]:
ensemble_block_2d = block_utils.block_z500_freq(block_meta,ensemble_ds,fout_dir,block_season,block_diag = '2D', file_opts = 'x')


-> block_z500_freq ->  Starting freq blocking calculation
--> set_file_name_and_check_existance ->   called with file_opts = x, Checking if file is already written
--> set_file_name_and_check_existance ->  file_data='/glade/work/bundy/mdtf/outdir/20250429/blocking_nb/block_2D_ERA5_nens.1_1995-1996_DJF.nc' File exists, reading in ...
--> block_file_read_write ->  Reading file  /glade/work/bundy/mdtf/outdir/20250429/blocking_nb/block_2D_ERA5_nens.1_1995-1996_DJF.nc
--> block_file_read_write ->  Done ...
-> block_z500_freq ->  Min/max blocking frequency for ensemble  ERA5  =  0.0 , 49.11080711354309
--> set_file_name_and_check_existance ->   called with file_opts = x, Checking if file is already written
--> set_file_name_and_check_existance ->  file_data='/glade/work/bundy/mdtf/outdir/20250429/blocking_nb/block_2D_MERRA_nens.1_1995-1996_DJF.nc' File exists, reading in ...
--> block_file_read_write ->  Reading file  /glade/work/bundy/mdtf/outdir/20250429/blocking_nb/block_2D_MERRA_nens.1_1

In [35]:
print(ens_names)


['ERA5', 'MERRA', 'CESM2', 'timeslice_mdtfv3']


In [36]:

for i in range(len(ens_names)):
    ens_name = ens_names[i]
    print(f"++++ Ensemble {i}: {ens_name}")
    print(f"      {ensemble_block_1d[ens_name].dims}")



++++ Ensemble 0: ERA5
      ('name', 'lon')
++++ Ensemble 1: MERRA
      ('name', 'lon')
++++ Ensemble 2: CESM2
      ('name', 'lon')
++++ Ensemble 3: timeslice_mdtfv3
      ('name', 'lon')


In [ ]:
block_figs.block_plot_1d(block_meta,ensemble_block_1d,block_season,fig_out=True,dir_fig=fig_dir,
                         fig_name=f"block_1d_freq_{block_season}.png")



##### Plot 2D blocking: Ensemble ave, 1 ensemble member, or all ensemble members 


In [ ]:
#Needs cartopy, add to env or find something else
# TODO Existing error here
block_figs.block_plot_2d(block_meta,ensemble_block_2d,block_season,fig_out=True,dir_fig=fig_dir,ens_plot='av',
                         fig_name=f"block_2d_freq_{block_season}.png")


##### Annual cycle of 1D blocking (Hovmoller: longitude x month)
All seasons, split like the original POD page: OBS, model ensembles, and each MDTF case.

In [ ]:
# Daily climatology (day of year x longitude) of the 1D blocking flag, all seasons
ensemble_block_ac = block_utils.block_z500_freq(block_meta,ensemble_ds,fout_dir,'ANNCYC',block_diag='1DAC',file_opts='x')


In [ ]:
ens_types = block_meta['Ensemble Type']
obs_list  = [e for e in block_meta.index if ens_types[e] == 'obs']
mod_list  = [e for e in block_meta.index if ens_types[e] == 'model']
mdtf_list = [e for e in block_meta.index if ens_types[e] == 'mdtf']

if obs_list:
    block_figs.block_plot_anncycle(block_meta, ensemble_block_ac, obs_list,
                                   'Blocking frequency annual cycle: OBS',
                                   dir_fig=os.path.join(pod_work_dir, 'obs'), fig_name='block_freq_anncycle.OBS.png')
if mod_list:
    block_figs.block_plot_anncycle(block_meta, ensemble_block_ac, mod_list,
                                   'Blocking frequency annual cycle: model ensembles',
                                   dir_fig=fig_dir, fig_name='block_freq_anncycle.ENSEMBLES.png')
for case_name in mdtf_list:
    block_figs.block_plot_anncycle(block_meta, ensemble_block_ac, [case_name],
                                   f'Blocking frequency annual cycle: {case_name}',
                                   dir_fig=fig_dir, fig_name=f'block_freq_anncycle.{case_name}.png')


In [ ]:
block_figs.block_plot_1d(block_meta,ensemble_block_1d,block_season,fig_out=True,dir_fig=fig_dir,
                         fig_name=f"block_1d_freq_{block_season}.png")


In [40]:
fout_dir

'/glade/work/bundy/mdtf/outdir/20250429/blocking_nb'

## Build the POD web page
Stand-alone runs build the MDTF POD page here, with the MDTF framework's own html code
(`src/standalone_html.py`). Framework runs skip this; the framework builds the page itself.

In [ ]:
make_html = True
if make_html and run.mode != "framework":
    page = standalone_html.make_standalone_pod_page(
        POD_NAME, pod_code_dir, run.work_dir, settings_dict, run.case_list,
        driver=os.path.join(pod_code_dir, NOTEBOOK), obs_data=run.obs_data,
        save_ps=bool(case_info.get("save_ps", False)),
        make_tar=bool(case_info.get("make_variab_tar", False)))
    print(f"POD web page: {page}")
    if run.mode == "interactive":
        print("(The 'Executed notebook' link works after a headless run, or after saving this notebook as html to "
              f"{os.path.join(pod_work_dir, 'blocking_main_esnb_ipynb.html')})")
